# 🛡️ X-IDS: Explainable Network Intrusion Detection System
### تدريب وتقييم نموذج كشف الاختراقات وشرح القرارات بـ Explainable AI

هذا الدفتر مجهز للعمل **مباشرة في VS Code أو على Google Colab**، ومبني وفقاً للقواعد والمعايير العلمية لمشروع **X-IDS** وداتا سيت **CIC-IDS2017 Improved**.

--- 
### 📌 مراحل المشروع:
1. **استدعاء المكتبات البرمجية** (`pandas`, `numpy`, `sklearn`, `xgboost`, `shap`).
2. **فحص واستكشاف البيانات** (`Shape`, `Info`, `Duplicates`, `Nulls`, `Infs`).
3. **تنظيف البيانات (Data Cleaning)**: استبدال القيم اللانهائية وملء القيم الفارغة وحذف التكرار.
4. **منع الـ Data Leakage**: حذف معرّفات الشبكة (`IPs`, `Ports`, `Timestamps`).
5. **معالجة فئات الـ Attempted**: التعامل السليم مع محاولات الهجوم غير المكتملة.
6. **فصل المدخلات والهدف ($X$ & $y$)** وعمل **Stratified Split** وحفظ التوزيع العادل.
7. **تدريب الموديلات الأساسية**: `Random Forest` و `XGBoost` مع معالجة الـ Class Imbalance.
8. **التقييم الشامل**: عدم الاكتفاء بالـ Accuracy، وقياس `Precision` و `Recall` و `F1-Score` و `ROC-AUC`.
9. **اختبار التعميم (Leave-One-Attack-Out)**: قياس قدرة الموديل على اكتشاف هجمات غير مسبوقة (Zero-Day).
10. **تفسير القرارات بـ SHAP**: توضيح أسباب كل قرار وتحليل أهم الخصائص المؤثرة.

## الخطوة 1: استدعاء المكتبات الأساسية

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

# XGBoost & SHAP
import xgboost as xgb
import shap

# إعدادات العرض
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('✅ تم استيراد جميع المكتبات بنجاح!')

## الخطوة 2: تحميل البيانات وفحص شكلها الأولي

In [ ]:
# مسار ملف البيانات (الملف التجريبي المولد أو ملف الداتا سيت الكامل)
DATA_PATH = 'data/cic_ids2017_sample.csv'

df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()  # إزالة المسافات الزائدة من أسماء الأعمدة

print(f"📊 أبعاد الداتا سيت: {df.shape[0]:,} صف × {df.shape[1]} عمود\n")
df.head()

## الخطوة 3: فحص جودة البيانات (القيم الفارغة، اللانهائية، والتكرار)

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
inf_count = np.isinf(df[num_cols]).sum().sum()
null_count = df.isnull().sum().sum()
dup_count = df.duplicated().sum()

print(f"🔍 عدد الصفوف المكررة: {dup_count:,}")
print(f"🔍 عدد القيم الفارغة (Nulls): {null_count:,}")
print(f"🔍 عدد قيم اللانهاية (Infs): {inf_count:,}")

print("\n🎯 توزيع الفئات في عمود Label:")
print(df['Label'].value_counts())

## الخطوة 4: تنظيف البيانات (Data Cleaning)
- استبدال الـ Infs بـ NaNs.
- تعويض الـ NaNs بالوسيط الحسابي (Median) لكل عمود رقمي.
- حذف الصفوف المكررة بالكامل.

In [ ]:
# 1. حذف التكرارات
df_clean = df.drop_duplicates().copy()

# 2. استبدال قيم inf بقيم nan
num_cols = df_clean.select_dtypes(include=[np.number]).columns
df_clean[num_cols] = df_clean[num_cols].replace([np.inf, -np.inf], np.nan)

# 3. ملء القيم الفارغة بالـ Median
for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("✅ تم تنظيف البيانات ومعالجة القيم الفارغة واللانهائية بنجاح!")
print(f"الصفوف المتبقية: {len(df_clean):,}")

## الخطوة 5: منع تسريب البيانات (Data Leakage) وحذف الأعمدة الثابتة
> **ملاحظة أمنية هامة (قاعدة X-IDS رقم 2):**
> أعمدة الـ Sockets مثل `Flow ID`, `Source IP`, `Destination IP`, `Timestamp`, `Source Port` تجعل النموذج يحفظ عناوين معينة أو أوقات معينة بدلاً من أن يتعلم الأنماط الحقيقية للهجوم، لذلك **يجب حذفها إجبارياً**.

In [ ]:
# أعمدة تسريب البيانات
leakage_cols = [
    'Flow ID', 'Source IP', 'Src IP', 'Destination IP', 'Dst IP',
    'Source Port', 'Src Port', 'Timestamp', 'Unnamed: 0'
]
cols_to_drop = [c for c in leakage_cols if c in df_clean.columns]
df_clean = df_clean.drop(columns=cols_to_drop)
print(f"🚫 تم حذف أعمدة التسريب: {cols_to_drop}")

# حذف الأعمدة الثابتة ذات التباين الصفري (Zero Variance)
numeric_features = df_clean.select_dtypes(include=[np.number])
constant_cols = [c for c in numeric_features.columns if numeric_features[c].std() == 0]
if constant_cols:
    df_clean = df_clean.drop(columns=constant_cols)
    print(f"🚫 تم حذف أعمدة ثابتة القيمة: {constant_cols}")

## الخطوة 6: معالجة تصنيفات Attempted
> **قاعدة X-IDS رقم 1:**
> محاولة الهجوم (Attempted) تعني أن المهاجم أرسل طلباً لكنه لم ينجح في اختراق النظام. وفقاً لورقة Engelen et al.، لا نعتبرها هجوماً ناجحاً. هنا سنعيد تصنيفها كـ `BENIGN`.

In [ ]:
df_clean['Label'] = df_clean['Label'].apply(lambda l: 'BENIGN' if 'Attempted' in str(l) else l)
print("🎯 توزيع الفئات بعد معالجة Attempted:")
print(df_clean['Label'].value_counts())

## الخطوة 7: استخراج $X$ و $y$ وتقسيم البيانات (Stratified Train-Test Split)

In [ ]:
# التصنيف الثنائي: 0 طبيعي (BENIGN) و 1 هجوم (Attack)
y = df_clean['Label'].apply(lambda l: 0 if str(l).strip().upper() == 'BENIGN' else 1).values
X = df_clean.drop(columns=['Label']).select_dtypes(include=[np.number])
feature_names = X.columns.tolist()

# تقسيم البيانات مع الحفاظ على النسب (Stratified Split)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# التقييس (StandardScaler) بالتدريب على بيانات التدريب فقط
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"Train samples: {X_train.shape[0]:,} | Test samples: {X_test.shape[0]:,} | Features: {X_train.shape[1]}")

## الخطوة 8: تدريب الموديلات الأساسية (Random Forest & XGBoost)

In [ ]:
# حساب أوزان الفئات للتعامل مع عدم التوازن (Class Imbalance)
neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)
scale_pos = neg_count / max(pos_count, 1)

print("🚀 تدريب Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

print("🚀 تدريب XGBoost Classifier...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(X_train, y_train)
print("✅ اكتمل تدريب النموذجين بنجاح!")

## الخطوة 9: التقييم الشامل (Accuracy, Precision, Recall, F1, ROC-AUC)

In [ ]:
def print_evaluation(model, name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n{'='*20} {name} Evaluation {'='*20}")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%")
    print(f"Precision: {precision_score(y_test, y_pred)*100:.2f}%")
    print(f"Recall:    {recall_score(y_test, y_pred)*100:.2f}%")
    print(f"F1-Score:  {f1_score(y_test, y_pred)*100:.2f}%")
    print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, digits=4))

print_evaluation(rf_model, "Random Forest")
print_evaluation(xgb_model, "XGBoost")

## الخطوة 10: رسم مصفوفة الارتباك (Confusion Matrix)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model, title in zip(axes, [rf_model, xgb_model], ["Random Forest", "XGBoost"]):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
    ax.set_title(f"{title} Confusion Matrix")
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.tight_layout()
plt.show()

## الخطوة 11: اختبار التعميم (Leave-One-Attack-Out Evaluation)
> استبعاد هجوم معين تماماً من التدريب، واختبار ما إذا كان الموديل سيكتشفه كـ Anomaly.

In [ ]:
UNSEEN_ATTACK = 'PortScan'

train_df = df_clean[~df_clean['Label'].str.contains(UNSEEN_ATTACK, case=False, na=False)]
test_df = df_clean[df_clean['Label'].str.contains(UNSEEN_ATTACK, case=False, na=False)]

y_tr = train_df['Label'].apply(lambda l: 0 if str(l).strip().upper() == 'BENIGN' else 1).values
X_tr = scaler.fit_transform(train_df.drop(columns=['Label']).select_dtypes(include=[np.number])[feature_names])

X_te = scaler.transform(test_df.drop(columns=['Label']).select_dtypes(include=[np.number])[feature_names])

loao_rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42)
loao_rf.fit(X_tr, y_tr)
preds = loao_rf.predict(X_te)
print(f"🎯 نسبة اكتشاف هجوم [{UNSEEN_ATTACK}] المستبعد دون تدريبه عليه: {np.mean(preds == 1)*100:.2f}%")

## الخطوة 12: تفسير قرارات الموديل بـ Explainable AI (SHAP)

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
X_sample = X_test[:100]
shap_values = explainer.shap_values(X_sample)

# رسم المخطط التفسيري لأهم الخصائص
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=True)